# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Parking-Lot Incident Review

### Scenario

You are the new visual-analytics intern at **Northgate Tech Park Security**. Last night the camera over
Parking Lot B recorded something, and the Head of Security wants a briefing **today**. You have the 2-minute
camera log (`unit3_ppt1_cctv_log.csv`), the camera still, and 10 s of microphone audio. Their message to you:

1. *"Before anyone asks for 'just store everything in full HD', tell me how much data our 400 cameras would
   produce per day. I need real numbers."*
2. *"Our night camera images always look murky. Can you prove what's wrong with them and fix one?"*
3. *"I am not watching two minutes of video. Give me a timeline with the moments that matter and one
   picture per moment."*
4. *"Something banged around 75 s. Was it a **gunshot** or **breaking glass**? Those trigger completely
   different responses."*
5. *"Put it all on ONE screen for the briefing."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer
in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the camera log, the camera still, a frame renderer, and the audio (plus two
**reference recordings** from the security vendor's sound library: a known gunshot and a known glass break).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cbook
from scipy import ndimage, signal

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

FPS, FS = 30, 16_000
cctv = pd.read_csv("unit3_ppt1_cctv_log.csv")
img = plt.imread(cbook.get_sample_data("grace_hopper.jpg")).astype(float)
lum = 0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]
night_still = 18 + 0.22 * lum + np.random.normal(0, 1.5, lum.shape)        # what the night camera delivers

FRAME_H, FRAME_W = 90, 160
def render_frame(row):
    yy, xx = np.mgrid[0:FRAME_H, 0:FRAME_W]
    base = row.mean_brightness * (0.75 + 0.25 * yy / FRAME_H)
    frame = np.dstack([base, base, base * 1.05]); frame[60:62, :, :] += 25
    if not np.isnan(row.car_x):
        frame[(np.abs(xx - row.car_x / 4) < 14) & (np.abs(yy - row.car_y / 4) < 6)] = [200, 40, 40]
    if not np.isnan(row.person_x):
        frame[((xx - row.person_x / 4) / 2.2) ** 2 + ((yy - row.person_y / 4) / 5) ** 2 <= 1] = [30, 90, 230]
    return np.clip(frame, 0, 255).astype(np.uint8)

def _clip(kind, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(int(2.0 * FS)) / FS
    x = 0.004 * rng.normal(size=t.size)
    m = t >= 0.5
    if kind == "gunshot":                       # sharp broadband crack + low boom, nothing after
        x[m] += 0.9 * np.exp(-(t[m] - 0.5) / 0.02) * rng.normal(size=m.sum())
        x[m] += 0.5 * np.exp(-(t[m] - 0.5) / 0.25) * np.sin(2 * np.pi * 70 * (t[m] - 0.5))
    else:                                       # impact + lingering high-frequency tinkle
        x[m] += 0.9 * np.exp(-(t[m] - 0.5) / 0.03) * rng.normal(size=m.sum())
        for _ in range(40):
            t0 = 0.55 + rng.exponential(0.35); f0 = rng.uniform(3000, 7000)
            mk = (t >= t0) & (t < t0 + 0.06)
            x[mk] += 0.12 * np.exp(-(t[mk] - t0) / 0.015) * np.sin(2 * np.pi * f0 * (t[mk] - t0))
    return t, x

ref_gunshot = _clip("gunshot", 11)
ref_glass = _clip("glass", 12)
rng = np.random.default_rng(0)
t_audio = 70 + np.arange(10 * FS) / FS
x_audio = 0.02 * np.sin(2 * np.pi * 120 * t_audio) + 0.004 * rng.normal(size=t_audio.size)
_, unk = _clip("glass", 99)
x_audio[(t_audio >= 74.5) & (t_audio < 76.5)] += unk[: int(2 * FS)]           # the mystery sound

print(f"Camera log: {len(cctv):,} frames | still: {night_still.shape} | audio: {x_audio.size:,} samples at {FS} Hz")
cctv.head()

## Task 1 — The Storage Reality Check (Request #1)

**Real-world usage:** Storage budgets for camera fleets are decided from exactly these back-of-envelope numbers.

**Why this technique:** Uncompressed storage = **W × H × C × B** per frame (Slides 25–28), frames = **f × T**
(Slide 35), audio samples = **f_s × T** (Slide 55).

**TODO:**
1. Compute the uncompressed size (in MB) of ONE 1920 × 1080 RGB frame at 8 bits per channel.
2. Compute the number of frames ONE camera records in 24 h at 30 fps.
3. Compute the total uncompressed video for 400 cameras for 24 h, in **terabytes** (1 TB = 1024⁴ bytes).
4. Compute the same total if the cameras were downsampled to 640 × 360 and 10 fps. What percentage of the
   pixels per frame remain after the downsample?

In [ ]:
# Task 1: uncompressed RGB, 8 bits/channel = 3 bytes/pixel
frame_bytes = 1920 * 1080 * 3
frame_mb = frame_bytes / 1_000_000
frames_per_day = 30 * 24 * 60 * 60
fleet_tb = (frame_bytes * frames_per_day * 400) / (1024**4)
small_frame_bytes = 640 * 360 * 3
fleet_tb_small = (small_frame_bytes * (10 * 24 * 60 * 60) * 400) / (1024**4)
pct_pixels_remaining = (640 * 360) / (1920 * 1080) * 100

print(f"One frame: {frame_mb:.2f} MB | frames/day: {frames_per_day:,}")
print(f"Fleet (1080p, 30 fps): {fleet_tb:,.1f} TB/day | Fleet (360p, 10 fps): {fleet_tb_small:,.1f} TB/day")
print(f"Pixels per frame remaining after downsample: {pct_pixels_remaining:.2f}%")


**Reflection — Answer:**

- **Spatial evidence:** fine details such as a person's face, small objects, or the exact shape of an item may disappear at 640 × 360.
- **Temporal evidence:** brief actions between frames, such as a quick hand movement or the exact order/timing of two people crossing, may be missed at 10 fps.


## Task 2 — Diagnose and Fix the Murky Night Image (Request #2)

**Real-world usage:** Before anyone interprets a night image, an analyst checks its histogram to see whether
the problem is exposure, contrast or clipping.

**Why this technique:** The **histogram** shows the intensity distribution (Slides 16–18); **Michelson
contrast** C = (L_max − L_min)/(L_max + L_min) quantifies it (Slide 15); a **contrast stretch** remaps the
occupied range onto 0–255.

**TODO:**
1. Plot `night_still` and its histogram (64 bins, range 0–255) side by side.
2. Compute its Michelson contrast using the 1st and 99th percentiles as L_min and L_max (robust to outliers).
3. Contrast-stretch it: map the 1st percentile → 0 and the 99th percentile → 255 (clip to 0–255).
4. Plot the stretched image + histogram, and print the new contrast.

In [ ]:
def michelson(lmax, lmin):
    denom = lmax + lmin
    return (lmax - lmin) / denom if denom else 0.0

lo, hi = np.percentile(night_still, [1, 99])
c_before = michelson(hi, lo)
stretched = np.clip((night_still - lo) * 255.0 / (hi - lo), 0, 255)
lo_after, hi_after = np.percentile(stretched, [1, 99])
c_after = michelson(hi_after, lo_after)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), gridspec_kw={"height_ratios": [2, 1]})
axes[0, 0].imshow(night_still, cmap="gray", vmin=0, vmax=255); axes[0, 0].set_title("Night image — original"); axes[0, 0].axis("off")
axes[1, 0].hist(night_still.ravel(), bins=64, range=(0, 255), color="gray"); axes[1, 0].set_title("Original histogram"); axes[1, 0].set_xlim(0, 255)
axes[0, 1].imshow(stretched, cmap="gray", vmin=0, vmax=255); axes[0, 1].set_title("Contrast-stretched image"); axes[0, 1].axis("off")
axes[1, 1].hist(stretched.ravel(), bins=64, range=(0, 255), color="gray"); axes[1, 1].set_title("Stretched histogram"); axes[1, 1].set_xlim(0, 255)
plt.tight_layout(); plt.show()
print(f"Contrast before: {c_before:.3f} -> after: {c_after:.3f}")


**Reflection — Answer:**

The original histogram is **narrow and concentrated toward the dark/low-intensity end**, which indicates a low-contrast, underexposed-looking image. Contrast stretching spreads the occupied intensity range across 0–255, making differences that were already present easier to see. It does not add new scene details; it only remaps existing pixel values and can also amplify noise. A histogram records how many pixels have each intensity but discards their locations, so different images can have the same histogram and the histogram alone cannot identify the objects in the image.


## Task 3 — From Two Minutes of Video to a One-Glance Timeline (Request #3)

**Real-world usage:** Security reviewers search for events; they don't watch footage linearly.

**Why this technique:** **Motion vectors → events → timeline + representative frames** (Slides 38–40, 45, 47).
Content-based keyframes beat uniform sampling when activity is sparse.

**TODO:**
1. Compute the person's speed in px/s as the magnitude of their motion vector over a 1-second window
   (`diff(FPS)` on x and y — single-frame differences are dominated by tracking jitter).
2. Find the time ranges where the person is **standing** (speed < 3 px/s) and **running** (speed > 25 px/s).
3. Detect the frames where `audio_level_db` first rises above −20 dB and where `mean_brightness` jumps by more
   than 40 in one frame.
4. Plot a timeline of `motion_energy` with a vertical marker + label for each event you found.
5. Show **one keyframe per event** (use `render_frame(cctv.iloc[i])`).

In [ ]:
# TODO 1: estimate speed over a 1-second window (30 frames).
# Coordinates are in the CSV's pixel coordinate system; missing detections stay NaN.
x = cctv["person_x"].astype(float)
y = cctv["person_y"].astype(float)
dx = x.diff(FPS)
dy = y.diff(FPS)
speed = np.sqrt(dx**2 + dy**2) / 1.0
# Require a detected person at both ends of the window.
valid = x.notna() & y.notna() & x.shift(FPS).notna() & y.shift(FPS).notna()
speed = speed.where(valid)

# TODO 2: print contiguous time ranges for standing and running.
def print_ranges(mask, label):
    mask = pd.Series(mask, index=cctv.index).fillna(False).astype(bool)
    groups = (mask != mask.shift(fill_value=False)).cumsum()
    found = False
    for _, ids in mask[mask].groupby(groups[mask]):
        start_i, end_i = int(ids.index[0]), int(ids.index[-1])
        print(f"{label}: {cctv.time_s.iloc[start_i]:.2f}–{cctv.time_s.iloc[end_i]:.2f} s")
        found = True
    if not found: print(f"{label}: no ranges detected")

print_ranges(speed < 3, "Standing (speed < 3 px/s)")
print_ranges(speed > 25, "Running (speed > 25 px/s)")

# TODO 3: first crossing above -20 dB; brightness jumps >40 between adjacent frames.
audio_hits = np.flatnonzero(cctv["audio_level_db"].to_numpy() > -20)
audio_event_idx = int(audio_hits[0]) if len(audio_hits) else None
brightness_delta = cctv["mean_brightness"].diff()
lighting_hits = np.flatnonzero((brightness_delta > 40).to_numpy())
lighting_event_idx = int(lighting_hits[0]) if len(lighting_hits) else None
print("Audio event frame:", audio_event_idx, "time:", None if audio_event_idx is None else f"{cctv.time_s.iloc[audio_event_idx]:.2f} s")
print("Lighting jump frame:", lighting_event_idx, "time:", None if lighting_event_idx is None else f"{cctv.time_s.iloc[lighting_event_idx]:.2f} s")

# TODO 4: annotated motion-energy timeline.
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(cctv.time_s, cctv.motion_energy, lw=0.8, label="motion energy")
if audio_event_idx is not None:
    ta = cctv.time_s.iloc[audio_event_idx]; ax.axvline(ta, color="tab:red", ls="--", label=f"Audio > -20 dB ({ta:.1f}s)")
if lighting_event_idx is not None:
    tl = cctv.time_s.iloc[lighting_event_idx]; ax.axvline(tl, color="tab:orange", ls=":", label=f"Brightness jump ({tl:.1f}s)")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Motion energy"); ax.set_title("CCTV event timeline"); ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

# TODO 5: one representative keyframe per detected event (and the 75 s sound event).
keyframes = []
if audio_event_idx is not None: keyframes.append(("Audio threshold crossing", audio_event_idx))
if lighting_event_idx is not None: keyframes.append(("Brightness jump", lighting_event_idx))
incident_idx = int(np.argmin(np.abs(cctv.time_s.to_numpy() - 75.0)))
keyframes.append(("Around 75 s incident", incident_idx))
fig, axes = plt.subplots(1, len(keyframes), figsize=(5 * len(keyframes), 3))
if len(keyframes) == 1: axes = [axes]
for ax, (label, i) in zip(axes, keyframes):
    ax.imshow(render_frame(cctv.iloc[i])); ax.set_title(f"{label}\n{cctv.time_s.iloc[i]:.2f} s"); ax.axis("off")
plt.tight_layout(); plt.show()


**Reflection — Answer:**

Uniform keyframes every 20 seconds could miss a brief audio spike, a sudden lighting change, or a short run that happens between sample times. The approach used here is **content-based keyframe selection / event-based summarization**: frames are selected because measurable activity or a change makes them important, rather than only because a fixed interval has elapsed.


## Task 4 — Gunshot or Glass? (Request #4)

**Real-world usage:** Acoustic event classification in security systems starts with an analyst comparing
spectrogram signatures against known references.

**Why this technique:** A **waveform** shows *when* something loud happened but not *what* it was; a
**spectrogram** (Slides 56–58) reveals each sound's time–frequency signature.

**TODO:**
1. Plot the waveforms of `ref_gunshot`, `ref_glass` and the mystery segment of `x_audio` (74.5–76.5 s).
2. Plot their spectrograms (use `signal.spectrogram(x, fs=FS, nperseg=512, noverlap=384)`, show in dB,
   y-axis 0–8000 Hz).
3. Compute, for each clip, the fraction of spectral energy **above 3 kHz** in the 0.2–1.2 s after the impact.
4. Print a one-line verdict.

In [ ]:
mystery = x_audio[(t_audio >= 74.5) & (t_audio < 76.5)]
clips = {"Reference: gunshot": ref_gunshot[1], "Reference: glass break": ref_glass[1], "MYSTERY (74.5-76.5 s)": mystery}

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
hf_fraction = {}
for col, (name, clip) in enumerate(clips.items()):
    local_t = np.arange(len(clip)) / FS
    axes[0, col].plot(local_t, clip, lw=0.45)
    axes[0, col].set_title(name); axes[0, col].set_xlabel("Time (s)"); axes[0, col].set_ylabel("Amplitude")
    f_spec, t_spec, Sxx = signal.spectrogram(clip, fs=FS, nperseg=512, noverlap=384)
    axes[1, col].pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx + 1e-12), shading="auto", vmin=-110, vmax=-40, cmap="magma")
    axes[1, col].set_ylim(0, 8000); axes[1, col].set_xlabel("Time (s)"); axes[1, col].set_ylabel("Frequency (Hz)")
    # Compare spectral energy 0.2–1.2 seconds after the impact (impact at 0.5 s).
    band = (t_spec >= 0.7) & (t_spec <= 1.7)
    high = f_spec >= 3000
    segment = Sxx[:, band]
    total_energy = segment.sum()
    hf_fraction[name] = float(segment[high, :].sum() / total_energy) if total_energy else 0.0
plt.tight_layout(); plt.show()

print("Fraction of spectral energy above 3 kHz (0.2–1.2 s after impact):")
for name, fraction in hf_fraction.items(): print(f"  {name}: {fraction:.3f}")

mystery_name = "MYSTERY (74.5-76.5 s)"
verdict = "breaking glass" if hf_fraction[mystery_name] > hf_fraction["Reference: gunshot"] else "gunshot"
print(f"Verdict: the mystery sound is more consistent with {verdict}; compare the lingering high-frequency energy with the reference spectrograms.")


**Reflection — Answer:**

No. The three waveforms can all show a sudden high-amplitude impact, so amplitude over time alone is not enough to classify the sound confidently. A spectrogram adds **time × frequency × magnitude**: it shows when each frequency occurs and how strong it is. The decision is supported by the **scattered high-frequency specks** that continue after the initial vertical impact stripe, matching the glass-break reference rather than the gunshot's rapidly decaying broadband crack and low boom.


## Task 5 — The One-Screen Briefing (Request #5)

**Real-world usage:** Incident briefings use a synchronized multimodal view so every claim is backed by two
modalities at the same timestamp.

**Why this technique:** **Image–video–audio integration** linked through a common time reference
(Slide 59), with **overview first** (timeline) and detail at the selected moment.

**TODO:** Build ONE figure with:
1. the rendered video frame at t = 75 s (left),
2. the full 120 s motion + audio timeline with a red marker at 75 s (top right),
3. the spectrogram of `x_audio` (70–80 s) with the same marker (bottom right),
4. a conclusion-style figure title (what happened, not "Incident Dashboard").

In [ ]:
t_sel = 75.0
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 2], height_ratios=[1, 1.3])
ax_frame = fig.add_subplot(gs[:, 0])
idx_sel = int(np.argmin(np.abs(cctv.time_s.to_numpy() - t_sel)))
ax_frame.imshow(render_frame(cctv.iloc[idx_sel])); ax_frame.axis("off")
ax_frame.set_title(f"Video frame at {cctv.time_s.iloc[idx_sel]:.1f} s")

ax_timeline = fig.add_subplot(gs[0, 1])
ax_timeline.plot(cctv.time_s, cctv.motion_energy, lw=0.7, label="motion energy")
ax_audio = ax_timeline.twinx()
ax_audio.plot(cctv.time_s, cctv.audio_level_db, lw=0.65, alpha=0.75, color="tab:green", label="audio level (dB)")
ax_timeline.axvline(t_sel, color="red", lw=2, label="selected moment: 75 s")
ax_timeline.set_xlim(0, 120); ax_timeline.set_xlabel("Time (s)"); ax_timeline.set_ylabel("Motion energy")
ax_audio.set_ylabel("Audio level (dB)"); ax_timeline.set_title("120-second CCTV overview")
lines1, labels1 = ax_timeline.get_legend_handles_labels(); lines2, labels2 = ax_audio.get_legend_handles_labels()
ax_timeline.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper right")

ax_spec = fig.add_subplot(gs[1, 1])
f_spec, tt_spec, Sxx_spec = signal.spectrogram(x_audio, fs=FS, nperseg=512, noverlap=384)
ax_spec.pcolormesh(tt_spec + t_audio[0], f_spec, 10 * np.log10(Sxx_spec + 1e-12), shading="auto", cmap="magma", vmin=-110, vmax=-40)
ax_spec.axvline(t_sel, color="cyan", lw=2); ax_spec.set_ylim(0, 8000)
ax_spec.set_xlabel("Time (s)"); ax_spec.set_ylabel("Frequency (Hz)"); ax_spec.set_title("Audio spectrogram (70–80 s); marker at 75 s")
fig.suptitle("Around 75 s: an impact with lingering high-frequency energy is consistent with breaking glass", fontsize=12)
plt.tight_layout(); plt.show()


**Reflection — Answer:**

If audio were offset by +2 seconds, the sound would appear to occur later than the corresponding video event. An analyst could incorrectly associate the bang with a different movement or frame, leading to a false conclusion. The **synchronized timeline and spectrogram panel** would reveal the mismatch because the red 75-second marker would no longer line up with the impact's time-frequency signature; accurate synchronization is essential when combining modalities.


**Final briefing note — Answer:**

At approximately **75 seconds**, the microphone records a sharp impact followed by lingering high-frequency energy consistent with **breaking glass**, while the selected CCTV frame and timeline provide visual context; this is an analytical indication, not definitive forensic identification. I would put the **synchronized multimodal view** first because it links the video frame, event timeline, and spectrogram at a common timestamp, following the lecture idea of moving from structure to pattern, change, event, and meaning. The spectrogram is important because its time × frequency × magnitude view distinguishes a short impact from the high-frequency tinkle associated with glass. For storage planning, the uncompressed 1080p/30-fps estimate demonstrates why the team should evaluate a documented compression/retention policy and retain higher-quality footage around detected events. If lower-resolution/10-fps recording is used for routine footage, preserve original-quality evidence around incidents so small spatial details and brief temporal actions are not permanently lost.
